In [1]:
!uv add requests python-frontmatter

Resolved 116 packages in 20ms
Checked 111 packages in 32ms


In [12]:
import requests
import io
import zipfile
import frontmatter

In [5]:
repo_owner='evidentlyai'
repo_name='docs'
branch_name='main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response=requests.get(zip_url)

In [7]:
zip_archive=zipfile.ZipFile(io.BytesIO(zip_response.content))

In [8]:
filenames=zip_archive.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [11]:
filename='docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_content =mdx_file.read().decode('utf-8')
print(mdx_content[:150])

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **


In [15]:
post=frontmatter.loads(mdx_content)
print(post.content[:100])
print(post.metadata)
_, filename_corrected = filename.split("/",maxsplit=1)
print(filename_corrected)

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En
{'title': 'Alerts', 'description': 'How to set up alerts.'}
docs/platform/alerts.mdx


In [18]:
docs={
    'content':post.content,
    'title':post.metadata.get('title'),
    'description':post.metadata.get('description'),
    'filename':filename_corrected
}

In [19]:
print(docs)

{'content': '<Check>\n  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n![](/images/alerts.png)\n\nTo enable alerts, open the Project and navigate to the "Alerts" in the left menu. You must set:\n\n* A notification channel.\n\n* An alert condition.\n\n## Notification channels\n\nYou can choose between the following options:\n\n* **Email**. Add email addresses to send alerts to.\n\n* **Slack**. Add a Slack webhook.\n\n* **Discord**. Add a Discord webhook.\n\n## Alert conditions\n\n### Failed tests\n\nIf you use Tests (conditional checks) in your Project, you can tie alerting to the failed Tests in a Test Suite. Toggle this option on the Alerts page. Evidently will set an alert to the defined channel if any of the Tests fail.\n\n<Tip>\n  **How to avoid alert fatigue?** Use the `is_critical` parameter to mark non-critical Test as Warnings. Setting it to `False` prevent alerts for those checks even if they fail.\n</Tip>\n\n

In [28]:
def read_github_repository(repo_owner,repo_name,branch="main"):
    url =f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response=requests.get(url)
    response.raise_for_status()

    documents=[]
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith(('.md','.mdx')):
                continue
            with zip_ref.open(file_path) as file:
                content=file.read().decode('utf-8')
                post = frontmatter.loads(content)
                doc={
                    'content':post.content,
                    'title':post.metadata.get('title'),
                    'descirption':post.metadata.get('description'),
                    'filename':file_path.split('/',1)[-1]
                }
                documents.append(doc)
    return documents

In [29]:
repo_owner = 'evidentlyai'
repo_name = 'docs'

documents = read_github_repository(repo_owner, repo_name)

print(f"Downloaded {len(documents)} documents")

Downloaded 95 documents


In [30]:
!uv add gitsource

Resolved 116 packages in 1ms
Checked 111 packages in 5ms


In [31]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [32]:
document = files[10]

print(document.filename)
print(document.content[:160])


docs/library/output_formats.mdx
---
title: 'Output formats'
description: 'How to export the evaluation results.'
---

You can view or export Reports in multiple formats.

**Pre-requisites**:




In [33]:
import frontmatter 

post = frontmatter.loads(document.content)
data = post.to_dict()
data['filename'] = document.filename


In [34]:
document.parse()

{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

In [35]:
documents = [f.parse() for f in files]